# 第1章 — Colabの実行環境

『セキュリティエンジニアのための機械学習』第1章に対応。
Colabの **ランタイム → すべてのセルを実行** で環境確認と最小学習を実行する。
無料のCPUランタイムで十分。GPU・TPU、Google Drive、APIキーは不要。
インストールは新しいランタイムの最初に実行する。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[
            "numpy==2.2.6",
            "pandas==2.2.3",
            "scipy==1.15.3",
            "scikit-learn==1.7.2",
            "matplotlib==3.10.3",
            "optuna==4.5.0",
            "lightgbm==4.6.0",
        ],
    ],
    check=True,
)

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt

print(f"Python {platform.python_version()} | CPU runtime")
for package in ("numpy", "pandas", "scipy", "scikit-learn", "optuna", "lightgbm"):
    print(f"{package}: {version(package)}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 作業領域とランタイム

Colabのファイルはセッション終了で失われる。第2章では結果をZIPにまとめて保存できる。
依存パッケージはノートブック内に固定し、実行したバージョンも結果に記録する。
ランタイムの制限・仕様は [Colab FAQ](https://research.google.com/colaboratory/faq.html) を参照。

In [ ]:
from pathlib import Path

workspace = Path.cwd() / "security-ml-lab"
workspace.mkdir(exist_ok=True)
probe = workspace / "runtime-check.txt"
probe.write_text("ready\n", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "ready\n"
probe.unlink()
print(f"Workspace: {workspace}")

## 学習 → 未知データによる評価

動作確認用の合成データを使う。セキュリティ上の検知性能を表すデータではない。
標準化は訓練データだけから学習し、テストデータへ適用する。

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=10, n_informative=6, random_state=101)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=101,
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=101))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(f"Synthetic holdout accuracy: {accuracy_score(y_test, prediction):.3f}")
ConfusionMatrixDisplay.from_predictions(y_test, prediction, colorbar=False)
plt.show()

第2章では実データの読み込み、重複・ラベルの監査、ハイパーパラメータ探索、
閾値選択、テスト評価を順に進める。第2章のノートブックも単独で実行できる。